In [1]:
from pyspark.sql import functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType,
    BooleanType,
    TimestampType
)

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 3, Finished, Available, Finished, False)

In [2]:
# ============================================================
# READ BRONZE TABLES
# ============================================================

stylehaus_bronze_df = spark.read.table(
    "bronze.StyleHausProducts"
)

urbanthread_bronze_df = spark.read.table(
    "bronze.UrbanThreadProducts"
)

print("Bronze tables loaded successfully.")

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 4, Finished, Available, Finished, False)

Bronze tables loaded successfully.


In [3]:
# ============================================================
# BRONZE SOURCE VALIDATION
# ============================================================

stylehaus_bronze_count = stylehaus_bronze_df.count()

urbanthread_bronze_count = urbanthread_bronze_df.count()

total_bronze_count = (
    stylehaus_bronze_count +
    urbanthread_bronze_count
)

print("StyleHaus Bronze Records:", stylehaus_bronze_count)

print("UrbanThread Bronze Records:", urbanthread_bronze_count)

print("Total Bronze Records:", total_bronze_count)

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 5, Finished, Available, Finished, False)

StyleHaus Bronze Records: 94
UrbanThread Bronze Records: 70
Total Bronze Records: 164


In [4]:
# ============================================================
# SILVER CANONICAL SCHEMA
# ============================================================

Silver_Product_Schema = StructType([

    StructField("SourceSystem", StringType(), True),

    StructField("SourceProductID", StringType(), True),

    StructField("ProductName", StringType(), True),

    StructField("Brand", StringType(), True),

    StructField("Category", StringType(), True),

    StructField("Description", StringType(), True),

    StructField("Price", DecimalType(10, 2), True),

    StructField("Color", StringType(), True),

    StructField("Size", StringType(), True),

    StructField("Material", StringType(), True),

    StructField("InStock", BooleanType(), True),

    StructField("UpdatedAt", TimestampType(), True),

    StructField("BatchID", StringType(), True),

    StructField("IngestionTimestamp", TimestampType(), True)

])

print("Silver canonical schema defined successfully.")

Silver_Product_Schema

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 6, Finished, Available, Finished, False)

Silver canonical schema defined successfully.


StructType([StructField('SourceSystem', StringType(), True), StructField('SourceProductID', StringType(), True), StructField('ProductName', StringType(), True), StructField('Brand', StringType(), True), StructField('Category', StringType(), True), StructField('Description', StringType(), True), StructField('Price', DecimalType(10,2), True), StructField('Color', StringType(), True), StructField('Size', StringType(), True), StructField('Material', StringType(), True), StructField('InStock', BooleanType(), True), StructField('UpdatedAt', TimestampType(), True), StructField('BatchID', StringType(), True), StructField('IngestionTimestamp', TimestampType(), True)])

In [5]:
# ============================================================
# COMMON HELPER FUNCTIONS
# ============================================================

def clean_string(column_name):

    value = F.trim(F.col(column_name))

    return (

        F.when(
            value == "",
            None
        )

        .otherwise(value)

    )


def normalize_text(column_name):

    return (

        F.when(
            F.col(column_name).isNull(),
            None
        )

        .otherwise(

            F.trim(

                F.regexp_replace(
                    F.col(column_name),
                    r"\s+",
                    " "
                )

            )

        )

    )


def normalize_mapping(column_name, mapping):

    mapping_expr = F.create_map(

        *[
            item

            for pair in mapping.items()

            for item in (
                F.lit(pair[0]),
                F.lit(pair[1])
            )

        ]

    )

    normalized_key = F.upper(
        F.trim(F.col(column_name))
    )

    return F.coalesce(

        mapping_expr[normalized_key],

        F.trim(F.col(column_name))

    )

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 7, Finished, Available, Finished, False)

In [6]:
# ============================================================
# STYLEHAUS TRANSFORMATION
# ============================================================

stylehaus_silver_df = (

    stylehaus_bronze_df

    .select(

        clean_string("SourceSystem")
        .alias("SourceSystem"),

        clean_string("sku")
        .alias("SourceProductID"),

        clean_string("product_name")
        .alias("ProductName"),

        clean_string("brand")
        .alias("Brand"),

        clean_string("category")
        .alias("Category"),

        clean_string("description")
        .alias("Description"),

        F.trim(F.col("price"))
        .cast("decimal(10,2)")
        .alias("Price"),

        clean_string("color")
        .alias("Color"),

        clean_string("size")
        .alias("Size"),

        clean_string("material")
        .alias("Material"),

        F.when(

            F.lower(F.trim(F.col("in_stock"))).isin(
                "true", "1", "yes"
            ),

            True

        )

        .when(

            F.lower(F.trim(F.col("in_stock"))).isin(
                "false", "0", "no"
            ),

            False

        )

        .otherwise(None)

        .cast("boolean")

        .alias("InStock"),

        F.to_timestamp(
            F.col("updated_at"),
            "yyyy-MM-dd HH:mm:ss"
        ).alias("UpdatedAt"),

        clean_string("BatchID")
        .alias("BatchID"),

        F.col("IngestionTimestamp")
        .cast("timestamp")
        .alias("IngestionTimestamp")

    )

)

print("StyleHaus transformation completed.")

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 8, Finished, Available, Finished, False)

StyleHaus transformation completed.


In [7]:
# ============================================================
# STYLEHAUS TRANSFORMATION
# ============================================================

stylehaus_silver_df = (

    stylehaus_bronze_df

    .select(

        clean_string("SourceSystem")
        .alias("SourceSystem"),

        clean_string("sku")
        .alias("SourceProductID"),

        clean_string("product_name")
        .alias("ProductName"),

        clean_string("brand")
        .alias("Brand"),

        clean_string("category")
        .alias("Category"),

        clean_string("description")
        .alias("Description"),

        F.trim(F.col("price"))
        .cast("decimal(10,2)")
        .alias("Price"),

        clean_string("color")
        .alias("Color"),

        clean_string("size")
        .alias("Size"),

        clean_string("material")
        .alias("Material"),

        F.when(

            F.lower(F.trim(F.col("in_stock"))).isin(
                "true", "1", "yes"
            ),

            True

        )

        .when(

            F.lower(F.trim(F.col("in_stock"))).isin(
                "false", "0", "no"
            ),

            False

        )

        .otherwise(None)

        .cast("boolean")

        .alias("InStock"),

        F.to_timestamp(
            F.col("updated_at"),
            "yyyy-MM-dd HH:mm:ss"
        ).alias("UpdatedAt"),

        clean_string("BatchID")
        .alias("BatchID"),

        F.col("IngestionTimestamp")
        .cast("timestamp")
        .alias("IngestionTimestamp")

    )

)

print("StyleHaus transformation completed.")


stylehaus_silver_df.select(

    F.count("*").alias("TotalRecords"),

    F.count(
        F.when(F.col("SourceProductID").isNull(), 1)
    ).alias("MissingProductID"),

    F.count(
        F.when(F.col("ProductName").isNull(), 1)
    ).alias("MissingProductName"),

    F.count(
        F.when(F.col("Price").isNull(), 1)
    ).alias("InvalidPrice"),

    F.count(
        F.when(F.col("UpdatedAt").isNull(), 1)
    ).alias("InvalidUpdatedAt"),

    F.count(
        F.when(F.col("InStock").isNull(), 1)
    ).alias("InvalidAvailability")

).show(truncate=False)

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 9, Finished, Available, Finished, False)

StyleHaus transformation completed.
+------------+----------------+------------------+------------+----------------+-------------------+
|TotalRecords|MissingProductID|MissingProductName|InvalidPrice|InvalidUpdatedAt|InvalidAvailability|
+------------+----------------+------------------+------------+----------------+-------------------+
|94          |0               |0                 |0           |0               |0                  |
+------------+----------------+------------------+------------+----------------+-------------------+



In [8]:
# ============================================================
# URBANTHREAD TRANSFORMATION
# ============================================================

urbanthread_silver_df = (

    urbanthread_bronze_df

    .select(

        clean_string("SourceSystem")
        .alias("SourceSystem"),

        clean_string("product_code")
        .alias("SourceProductID"),

        clean_string("title")
        .alias("ProductName"),

        clean_string("manufacturer")
        .alias("Brand"),

        clean_string("type")
        .alias("Category"),

        clean_string("details")
        .alias("Description"),

        F.trim(F.col("retail_price"))
        .cast("decimal(10,2)")
        .alias("Price"),

        clean_string("colour")
        .alias("Color"),

        clean_string("dimensions")
        .alias("Size"),

        clean_string("fabric")
        .alias("Material"),

        F.when(

            F.lower(F.trim(F.col("availability"))).isin(
                "in stock", "true", "1", "yes"
            ),

            True

        )

        .when(

            F.lower(F.trim(F.col("availability"))).isin(
                "out of stock", "false", "0", "no"
            ),

            False

        )

        .otherwise(None)

        .cast("boolean")

        .alias("InStock"),

        F.to_timestamp(
            F.col("last_modified"),
            "yyyy-MM-dd HH:mm:ss"
        ).alias("UpdatedAt"),

        clean_string("BatchID")
        .alias("BatchID"),

        F.col("IngestionTimestamp")
        .cast("timestamp")
        .alias("IngestionTimestamp")

    )

)

print("UrbanThread transformation completed.")

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 10, Finished, Available, Finished, False)

UrbanThread transformation completed.


In [9]:
# ============================================================
# URBANTHREAD VALIDATION
# ============================================================

print(
    "Bronze UrbanThread Count:",
    urbanthread_bronze_df.count()
)

print(
    "Silver UrbanThread Count:",
    urbanthread_silver_df.count()
)

print("\nURBANTHREAD SCHEMA")

urbanthread_silver_df.printSchema()

print("\nURBANTHREAD SAMPLE RECORDS")

urbanthread_silver_df.show(
    10,
    truncate=False
)

urbanthread_silver_df.select(

    F.count("*").alias("TotalRecords"),

    F.count(
        F.when(F.col("SourceProductID").isNull(), 1)
    ).alias("MissingProductID"),

    F.count(
        F.when(F.col("ProductName").isNull(), 1)
    ).alias("MissingProductName"),

    F.count(
        F.when(F.col("Price").isNull(), 1)
    ).alias("InvalidPrice"),

    F.count(
        F.when(F.col("UpdatedAt").isNull(), 1)
    ).alias("InvalidUpdatedAt"),

    F.count(
        F.when(F.col("InStock").isNull(), 1)
    ).alias("InvalidAvailability")

).show(truncate=False)

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 11, Finished, Available, Finished, False)

Bronze UrbanThread Count: 70
Silver UrbanThread Count: 70

URBANTHREAD SCHEMA
root
 |-- SourceSystem: string (nullable = true)
 |-- SourceProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Price: decimal(10,2) (nullable = true)
 |-- Color: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Material: string (nullable = true)
 |-- InStock: boolean (nullable = true)
 |-- UpdatedAt: timestamp (nullable = true)
 |-- BatchID: string (nullable = true)
 |-- IngestionTimestamp: timestamp (nullable = true)


URBANTHREAD SAMPLE RECORDS
+------------+---------------+----------------------------------+--------------+---------------------+-----------------------------------------------------------------------------+------+-----------+-----+-------------------+-------+-------------------+------------------------------------+-----------

In [10]:
# ============================================================
# BRAND STANDARDIZATION MAPPING
# ============================================================

Brand_Mapping = {

    "A.P.C.": "A.P.C.",

    "AMI PARIS": "AMI Paris",

    "ASICS": "ASICS",

    "ACNE STUDIOS": "Acne Studios",

    "ADIDAS": "Adidas",

    "ALLSAINTS": "AllSaints",

    "ARC'TERYX": "Arc'teryx",

    "BARBOUR": "Barbour",

    "BIRKENSTOCK": "Birkenstock",

    "BROOKS": "Brooks",

    "BURBERRY": "Burberry",

    "C.P. COMPANY": "C.P. Company",

    "CALVIN KLEIN": "Calvin Klein",

    "CANADA GOOSE": "Canada Goose",

    "CARHARTT": "Carhartt",

    "CARHARTT WIP": "Carhartt WIP",

    "CHAMPION": "Champion",

    "CLARKS": "Clarks",

    "COLE HAAN": "Cole Haan",

    "COLUMBIA": "Columbia",

    "CONVERSE": "Converse",

    "COS": "COS",

    "DIADORA": "Diadora",

    "DICKIES": "Dickies",

    "DR MARTENS": "Dr. Martens",

    "ECCO": "ECCO",

    "FILA": "FILA",

    "FILLING PIECES": "Filling Pieces",

    "FJALLRAVEN": "Fjallraven",

    "FOSSIL": "Fossil",

    "FRED PERRY": "Fred Perry",

    "GANT": "Gant",

    "GAP": "Gap",

    "GUCCI": "Gucci",

    "H&M": "H&M",

    "HOKA": "HOKA",

    "HOKA ONE ONE": "HOKA",

    "HANES": "Hanes",

    "HERSCHEL": "Herschel",

    "HUGO BOSS": "Hugo Boss",

    "JACK WOLFSKIN": "Jack Wolfskin",

    "LACOSTE": "Lacoste",

    "LEVI'S": "Levi's",

    "LEVIS": "Levi's",

    "LULULEMON": "Lululemon",

    "MAISON KITSUNE": "Maison Kitsune",

    "MANGO": "Mango",

    "MERRELL": "Merrell",

    "MIZUNO": "Mizuno",

    "MONCLER": "Moncler",

    "NEW BALANCE": "New Balance",

    "NIKE": "Nike",

    "NORSE PROJECTS": "Norse Projects",

    "NUDIE JEANS": "Nudie Jeans",

    "OAKLEY": "Oakley",

    "ON": "On",

    "OSPREY": "Osprey",

    "PATAGONIA": "Patagonia",

    "PUMA": "Puma",

    "RALPH LAUREN": "Ralph Lauren",

    "RAY-BAN": "Ray-Ban",

    "REEBOK": "Reebok",

    "SALOMON": "Salomon",

    "SAUCONY": "Saucony",

    "SCOTCH & SODA": "Scotch & Soda",

    "STONE ISLAND": "Stone Island",

    "STUSSY": "Stussy",

    "SUPERGA": "Superga",

    "TED BAKER": "Ted Baker",

    "THE NORTH FACE": "The North Face",

    "TIMBERLAND": "Timberland",

    "TOMMY HILFIGER": "Tommy Hilfiger",

    "UGG": "UGG",

    "UNDER ARMOUR": "Under Armour",

    "UNIQLO": "Uniqlo",

    "VANS": "Vans",

    "WOOLRICH": "Woolrich",

    "WRANGLER": "Wrangler",

    "ZARA": "Zara"

}

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 12, Finished, Available, Finished, False)

In [11]:
# ============================================================
# CATEGORY STANDARDIZATION MAPPING
# ============================================================

Category_Mapping = {

    "RUNNING SHOES - MEN":
        "Men's Running Shoes",

    "LIFESTYLE SHOES - MEN":
        "Men's Lifestyle Shoes",

    "LIFESTYLE SHOES - UNISEX":
        "Unisex Lifestyle Shoes",

    "LIFESTYLE - UNISEX":
        "Unisex Lifestyle Shoes",

    "LIFESTYLE - WOMEN":
        "Women's Lifestyle Shoes",

    "JEANS - MEN":
        "Men's Jeans",

    "OUTERWEAR - MEN":
        "Men's Outerwear",

    "FLEECE - MEN":
        "Men's Fleece",

    "PANTS - MEN":
        "Men's Pants",

    "T-SHIRTS - MEN":
        "Men's T-Shirts",

    "SHIRTS - MEN":
        "Men's Shirts",

    "KNITWEAR - MEN":
        "Men's Knitwear",

    "HOODIES - MEN":
        "Men's Hoodies",

    "BOOTS - MEN":
        "Men's Boots",

    "BOOTS - WOMEN":
        "Women's Boots",

    "BOOTS - UNISEX":
        "Unisex Boots",

    "BASKETBALL SHOES - MEN":
        "Men's Basketball Shoes",

    "TRAIL SHOES - MEN":
        "Men's Trail Shoes",

    "SKATE SHOES - UNISEX":
        "Unisex Skate Shoes",

    "SANDALS - UNISEX":
        "Unisex Sandals",

    "BASE LAYER - MEN":
        "Men's Base Layer",

    "TRAINING SHOES - MEN":
        "Men's Training Shoes",

    "HIKING SHOES - MEN":
        "Men's Hiking Shoes",

    "BAGS & ACCESSORIES":
        "Accessories",

    "SUNGLASSES & ACCESSORIES":
        "Accessories",

    "TOPS - MEN":
        "Men's Tops"

}

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 13, Finished, Available, Finished, False)

In [12]:
# ============================================================
# COLOR STANDARDIZATION MAPPING
# ============================================================

Color_Mapping = {

    # White variations
    "WHITE/BLK": "White/Black",
    "WHITE/BLACK": "White/Black",
    "WHITE/GREEN": "White/Green",
    "WHITE/ORANGE": "White/Orange",
    "OPTICAL WHITE": "Optical White",
    "CLOUD WHITE": "Cloud White",
    "ALL WHITE": "All White",

    # Black variations
    "BLACK/WHITE": "Black/White",
    "BLACK/DARK GREY": "Black/Dark Grey",
    "BLACK/GREEN": "Black/Green",
    "BLACK/GREY/WHITE": "Black/Grey/White",
    "BLACK/PHANTOM": "Black/Phantom",
    "BLACK/RED": "Black/Red",
    "MATTE BLACK": "Matte Black",
    "CORE BLACK": "Core Black",

    # Grey variations
    "GREY/WHITE": "Grey/White",
    "GREY/BLUE": "Grey/Blue",
    "GREY HEATHER": "Heather Grey",
    "HEATHER GREY": "Heather Grey",
    "CHARCOAL": "Charcoal",
    "OXFORD GREY": "Oxford Grey",
    "PITCH GREY": "Pitch Grey",
    "SLATE GREY": "Slate Grey",
    "SHEET ROCK/GREY": "Sheet Rock/Grey",
    "STONEWASH GREY": "Stonewash Grey",

    # Blue variations
    "BLUE/WHITE": "Blue/White",
    "BLUE/NIGHTLIFE": "Blue/Nightlife",
    "BLUE/SILVER": "Blue/Silver",
    "DARK MOUNTAIN BLUE": "Dark Mountain Blue",
    "COLLEGE BLUE": "College Blue",
    "THUNDER BLUE": "Thunder Blue",
    "VALLEY VIEW BLUE": "Valley View Blue",
    "NAVY BLUE": "Navy Blue",
    "NAVY/WHITE": "Navy/White",
    "NAVY BLAZER": "Navy Blazer",

    # Green variations
    "GOLD/GREEN": "Gold/Green",
    "GREEN/WHITE": "Green/White",
    "DARK GREEN": "Dark Green",
    "OLIVE GREEN": "Olive Green",
    "SAGE GREEN": "Sage Green",
    "SPRUCE GREEN": "Spruce Green",

    # Red variations
    "RED/BLACK": "Red/Black",
    "RED CHECK": "Red Check",
    "TNF RED": "TNF Red",
    "VARSITY RED/BLACK": "Varsity Red/Black",

    # Brown variations
    "HABANA": "Habana Brown",
    "BRITISH TAN": "British Tan",
    "BROWN DUCK": "Brown Duck",
    "BROWN/SILVER": "Brown/Silver",
    "EARTH BROWN": "Earth Brown",
    "WHEAT": "Wheat",
    "WHEAT NUBUCK": "Wheat Nubuck",
    "ARCHIVE BEIGE": "Archive Beige",

    # Other colors
    "INDIGO": "Indigo Rinse",
    "NAVY": "Navy",
    "OLIVE": "Olive",
    "KHAKI": "Khaki",
    "BEIGE": "Beige",
    "CREAM": "Cream",
    "CHESTNUT": "Chestnut",
    "TAUPE SUEDE": "Taupe Suede",
    "SAND SUEDE": "Sand Suede",
    "SEA SALT/CREAM": "Sea Salt/Cream",
    "DRY ECRU": "Dry Ecru",
    "MEDIUM WASH": "Medium Wash",
    "MEDIUM STONEWASH": "Medium Stonewash",
    "ANTIQUE WASH": "Antique Wash",
    "RAW INDIGO": "Raw Indigo",
    "FLAMINGO PINK": "Flamingo Pink",
    "METALLIC SILVER": "Metallic Silver",
    "SILVER/BLUE": "Silver/Blue",
    "TNF BLACK": "TNF Black"

}

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 14, Finished, Available, Finished, False)

In [13]:
# ============================================================
# COMMON CLEANSING FUNCTIONS
# ============================================================

def normalize_color(column_name):

    normalized_key = F.upper(
        F.trim(F.col(column_name))
    )

    mapping_expr = F.create_map(

        *[
            item

            for pair in Color_Mapping.items()

            for item in (
                F.lit(pair[0]),
                F.lit(pair[1])
            )

        ]

    )

    return (

        F.when(
            F.col(column_name).isNull(),
            None
        )

        .otherwise(

            F.coalesce(

                mapping_expr[normalized_key],

                F.initcap(
                    F.lower(
                        F.trim(F.col(column_name))
                    )
                )

            )

        )

    )


def apply_common_cleansing(df):

    return (

        df

        .withColumn(
            "Brand",
            normalize_mapping("Brand", Brand_Mapping)
        )

        .withColumn(
            "Category",
            normalize_mapping("Category", Category_Mapping)
        )

        .withColumn(
            "Color",
            normalize_color("Color")
        )

        .withColumn(
            "ProductName",
            normalize_text("ProductName")
        )

        .withColumn(
            "Description",
            normalize_text("Description")
        )

        .withColumn(
            "Material",
            normalize_text("Material")
        )

        .withColumn(
            "Size",
            normalize_text("Size")
        )

    )

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 15, Finished, Available, Finished, False)

In [14]:
# ============================================================
# STYLEHAUS COMMON CLEANSING
# ============================================================

stylehaus_clean_df = apply_common_cleansing(
    stylehaus_silver_df
)

print("StyleHaus cleansing completed.")

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 16, Finished, Available, Finished, False)

StyleHaus cleansing completed.


In [15]:
# ============================================================
# URBANTHREAD COMMON CLEANSING
# ============================================================

urbanthread_clean_df = apply_common_cleansing(
    urbanthread_silver_df
)

print("UrbanThread cleansing completed.")

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 17, Finished, Available, Finished, False)

UrbanThread cleansing completed.


In [16]:
# ============================================================
# TEMPORARY DATAFRAME FOR STANDARDIZATION VALIDATION
# ============================================================

all_clean_df = (

    stylehaus_clean_df

    .unionByName(
        urbanthread_clean_df
    )

)

print("Temporary combined DataFrame created.")
all_clean_df.select(
    "Brand"
).distinct().orderBy(
    "Brand"
).show(200, truncate=False)
brand_variants_df = (

    all_clean_df

    .groupBy(

        F.upper(
            F.trim(F.col("Brand"))
        ).alias("BrandKey")

    )

    .agg(

        F.collect_set("Brand").alias("BrandVariants")

    )

    .filter(

        F.size("BrandVariants") > 1

    )

)

brand_variants_df.show(
    100,
    truncate=False
)
all_clean_df.select(
    "Color"
).distinct().orderBy(
    "Color"
).show(200, truncate=False)

all_clean_df.select(
    "Category"
).distinct().orderBy(
    "Category"
).show(200, truncate=False)

print(
    "StyleHaus Clean Records:",
    stylehaus_clean_df.count()
)

print(
    "UrbanThread Clean Records:",
    urbanthread_clean_df.count()
)

print(
    "Combined Records:",
    all_clean_df.count()
)

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 18, Finished, Available, Finished, False)

Temporary combined DataFrame created.
+--------------+
|Brand         |
+--------------+
|A.P.C.        |
|AMI Paris     |
|ASICS         |
|Acne Studios  |
|Adidas        |
|AllSaints     |
|Arc'teryx     |
|Barbour       |
|Birkenstock   |
|Brooks        |
|Burberry      |
|C.P. Company  |
|COS           |
|Calvin Klein  |
|Canada Goose  |
|Carhartt      |
|Carhartt WIP  |
|Champion      |
|Clarks        |
|Cole Haan     |
|Columbia      |
|Converse      |
|Diadora       |
|Dickies       |
|Dr. Martens   |
|ECCO          |
|FILA          |
|Filling Pieces|
|Fjallraven    |
|Fossil        |
|Fred Perry    |
|Gant          |
|Gap           |
|Gucci         |
|H&M           |
|HOKA          |
|Hanes         |
|Herschel      |
|Hugo Boss     |
|Jack Wolfskin |
|Lacoste       |
|Levi's        |
|Lululemon     |
|Maison Kitsune|
|Mango         |
|Merrell       |
|Mizuno        |
|Moncler       |
|New Balance   |
|Nike          |
|Norse Projects|
|Nudie Jeans   |
|Oakley        |
|On       

In [17]:
# ============================================================
# FINAL COMBINATION OF SOURCE RECORDS
# ============================================================

all_clean_df = (

    stylehaus_clean_df

    .unionByName(
        urbanthread_clean_df
    )

)

print("Both source datasets combined successfully.")

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 19, Finished, Available, Finished, False)

Both source datasets combined successfully.


In [18]:
# ============================================================
# SCHEMA VALIDATION
# ============================================================

if stylehaus_clean_df.schema == urbanthread_clean_df.schema:

    print("SUCCESS: Source schemas are identical.")

else:

    raise Exception(
        "ERROR: Source schema mismatch detected."
    )

# ============================================================
# RECORD COUNT VALIDATION
# ============================================================

stylehaus_count = stylehaus_clean_df.count()

urbanthread_count = urbanthread_clean_df.count()

combined_count = all_clean_df.count()

print("StyleHaus:", stylehaus_count)

print("UrbanThread:", urbanthread_count)

print("Combined:", combined_count)

assert combined_count == (
    stylehaus_count + urbanthread_count
), "Record reconciliation failed!"

print("Record reconciliation passed.")    

all_clean_df.groupBy(
    "SourceSystem"
).count().show()

# ============================================================
# SOURCE PRODUCT ID DUPLICATE CHECK
# ============================================================

duplicate_source_ids_df = (

    all_clean_df

    .groupBy(
        "SourceSystem",
        "SourceProductID"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

)

duplicate_source_count = duplicate_source_ids_df.count()

print(
    "Duplicate Source IDs:",
    duplicate_source_count
)

duplicate_source_ids_df.show(
    truncate=False
)

assert duplicate_source_count == 0, (
    "Duplicate source identifiers detected!"
)
all_clean_df.show(
    20,
    truncate=False
)

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 20, Finished, Available, Finished, False)

SUCCESS: Source schemas are identical.
StyleHaus: 94
UrbanThread: 70
Combined: 164
Record reconciliation passed.
+------------+-----+
|SourceSystem|count|
+------------+-----+
|   StyleHaus|   94|
| UrbanThread|   70|
+------------+-----+

Duplicate Source IDs: 0
+------------+---------------+-----+
|SourceSystem|SourceProductID|count|
+------------+---------------+-----+
+------------+---------------+-----+

+------------+--------------------+----------------------------------------+--------------+---------------------+--------------------------------------------------------------------------------------------------------------------------------------------+------+----------------+-----+----------------------+-------+-------------------+------------------------------------+--------------------------+
|SourceSystem|SourceProductID     |ProductName                             |Brand         |Category             |Description                                                               

In [19]:
# ============================================================
# WRITE SILVER DELTA TABLE
# ============================================================

Silver_Table_Name = "Silver_Products"

(

    all_clean_df

    .write

    .format("delta")

    .mode("overwrite")

    .option("overwriteSchema", "true")

    .saveAsTable(Silver_Table_Name)

)

print(
    f"Successfully created Silver table: {Silver_Table_Name}"
)

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 21, Finished, Available, Finished, False)

Successfully created Silver table: Silver_Products


In [20]:
# ============================================================
# READ PERSISTED SILVER TABLE
# ============================================================

silver_products_df = spark.read.table(
    "Silver.Products"
)

print("Silver table loaded successfully.")

silver_count = silver_products_df.count()

print(
    "Silver Products Count:",
    silver_count
)

assert silver_count == 164, (
    f"Unexpected Silver record count: {silver_count}"
)

print("Silver record count validation passed.")

silver_products_df.printSchema()

silver_products_df.groupBy(
    "SourceSystem"
).count().show()

silver_products_df.show(
    20,
    truncate=False
)

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 22, Finished, Available, Finished, False)

Silver table loaded successfully.
Silver Products Count: 164
Silver record count validation passed.
root
 |-- SourceSystem: string (nullable = true)
 |-- SourceProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Price: decimal(10,2) (nullable = true)
 |-- Color: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Material: string (nullable = true)
 |-- InStock: boolean (nullable = true)
 |-- UpdatedAt: timestamp (nullable = true)
 |-- BatchID: string (nullable = true)
 |-- IngestionTimestamp: timestamp (nullable = true)

+------------+-----+
|SourceSystem|count|
+------------+-----+
|   StyleHaus|   94|
| UrbanThread|   70|
+------------+-----+

+------------+--------------------+----------------------------------------+--------------+---------------------+-------------------------------------------------------------------

In [21]:
# ============================================================
# PHASE 3 FINAL RECONCILIATION
# ============================================================

bronze_stylehaus_count = spark.read.table(
    "Bronze_StyleHausProducts"
).count()

bronze_urbanthread_count = spark.read.table(
    "Bronze_UrbanThreadProducts"
).count()

silver_final_count = spark.read.table(
    "Silver_Products"
).count()


print("======================================")

print("PHASE 3 - SILVER TRANSFORMATION SUMMARY")

print("======================================")

print(
    "Bronze StyleHaus:",
    bronze_stylehaus_count
)

print(
    "Bronze UrbanThread:",
    bronze_urbanthread_count
)

print(
    "Total Bronze Records:",
    bronze_stylehaus_count + bronze_urbanthread_count
)

print("--------------------------------------")

print(
    "Silver Products:",
    silver_final_count
)

print("--------------------------------------")

print(
    "Records Reconciliation:",
    "PASSED"
    if silver_final_count == (
        bronze_stylehaus_count + bronze_urbanthread_count
    )
    else "FAILED"
)

print("======================================")

StatementMeta(, 427b001e-195b-492a-b8fa-6e83908ecca5, 23, Finished, Available, Finished, False)

PHASE 3 - SILVER TRANSFORMATION SUMMARY
Bronze StyleHaus: 94
Bronze UrbanThread: 70
Total Bronze Records: 164
--------------------------------------
Silver Products: 164
--------------------------------------
Records Reconciliation: PASSED
